In [0]:
import dlt

@dlt.table(
  name="sales_bronze",
  comment="Raw sales ingestion"
)
def sales_bronze():
    return spark.read.format("json").load("/mnt/raw/sales")

In [0]:
import dlt

@dlt.table(
    name="dev.bronze.sales_bronze",
    comment="Raw sales data"
)
def sales_bronze():
    return spark.read.json("/mnt/raw/sales")

### Bronze Layer (Raw Ingestion + Basic Validation)

In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="analytics.bronze.sales_bronze",
    comment="Raw sales ingestion"
)
@dlt.expect("valid_order_id", "order_id IS NOT NULL")
@dlt.expect("valid_product", "product IS NOT NULL")
def sales_bronze():
    return (
        spark.read.format("json")
        .load("/mnt/raw/sales")
    )

### Explicit Schema Definition
Sometimes production pipelines require strict schema control.
You define it using StructType.

In [0]:
import dlt
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True)
])

@dlt.table(name="analytics.bronze.sales_bronze")
def sales_bronze():
    return (
        spark.read
        .schema(schema)
        .json("/mnt/raw/sales")
    )

### Schema in Views
DLT also supports views for intermediate transformations.

In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.view
def sales_clean_view():
    df = spark.read.table("dev.bronze.sales_bronze")

    return df.select(
        col("order_id"),
        col("product"),
        col("amount").cast("double"),
        col("customer_id")
    )

### Bronze Table With Schema Evolution

In [0]:
import dlt

@dlt.table(name="dev.bronze.sales_bronze")
def sales_bronze():
    return (
        spark.read
        .format("json")
        .option("mergeSchema","true")
        .load("/mnt/raw/sales")
    )

### Slowly changing dimension
### Bronze Layer (Raw Ingestion)
Bronze should only ingest raw data with minimal validation.

In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="dev.bronze.sales_bronze",
    comment="Raw sales ingestion"
)
@dlt.expect("valid_order_id", "order_id IS NOT NULL")
@dlt.expect("valid_product", "product IS NOT NULL")
def sales_bronze():
    return (
        spark.read.format("json")
        .load("/mnt/raw/sales")
        .withColumn("ingestion_time", current_timestamp())
    )